# Retinal Disease Classification using Vision Transformers

This notebook trains Vision Transformer (ViT), ResNet-50, and EfficientNet-B4 models for diabetic retinopathy classification.

**Dataset:** APTOS 2019 Blindness Detection

**Make sure to:**
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Upload your Kaggle API key (kaggle.json)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q timm albumentations tensorboard kaggle

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download Dataset from Kaggle

In [ ]:
# Upload your kaggle.json file
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

In [ ]:
# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download APTOS 2019 dataset
!kaggle competitions download -c aptos2019-blindness-detection
!unzip -q aptos2019-blindness-detection.zip -d data/
!ls data/

## 3. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)

# Set random seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 4. Configuration

In [ ]:
# Configuration - QUICK TEST MODE (5 epochs)
class Config:
    # Data
    data_dir = 'data'
    image_size = 224
    num_classes = 5
    
    # Model - Using ResNet50 for faster testing
    model_name = 'resnet50'  # Options: vit_base_patch16_224, resnet50, efficientnet_b4
    pretrained = True
    dropout = 0.2
    
    # Training - REDUCED FOR QUICK TEST
    batch_size = 32
    num_epochs = 5  # Quick test (change to 50 for full training)
    learning_rate = 3e-4
    weight_decay = 0.01
    warmup_epochs = 1  # Reduced warmup
    
    # Loss
    focal_gamma = 2.0
    label_smoothing = 0.1
    
    # Augmentation
    mixup_alpha = 0.2
    
    # Early stopping
    patience = 3  # Reduced for quick test
    
    # Class names
    class_names = ['No_DR', 'Mild_DR', 'Moderate_DR', 'Severe_DR', 'Proliferative_DR']

config = Config()
print("=" * 50)
print("QUICK TEST MODE - 5 EPOCHS")
print("=" * 50)
print(f"Model: {config.model_name}")
print(f"Epochs: {config.num_epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.learning_rate}")

## 5. Load and Explore Data

In [ ]:
# Load training data
train_df = pd.read_csv(f'{config.data_dir}/train.csv')
print(f"Total samples: {len(train_df)}")
print(f"\nClass distribution:")
print(train_df['diagnosis'].value_counts().sort_index())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=train_df, x='diagnosis')
ax.set_xticklabels(config.class_names)
plt.title('Class Distribution - APTOS 2019')
plt.xlabel('DR Severity')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Create image paths
train_df['image_path'] = train_df['id_code'].apply(
    lambda x: f"{config.data_dir}/train_images/{x}.png"
)

# Verify images exist
train_df['exists'] = train_df['image_path'].apply(os.path.exists)
print(f"Images found: {train_df['exists'].sum()} / {len(train_df)}")

In [ ]:
# Show sample images
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (idx, row) in enumerate(train_df.groupby('diagnosis').first().iterrows()):
    img = cv2.imread(row['image_path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(f"{config.class_names[idx]}")
    axes[i].axis('off')
plt.suptitle('Sample Images by Class')
plt.tight_layout()
plt.show()

## 6. Data Splits

In [ ]:
# Stratified train/val/test split
train_data, temp_data = train_test_split(
    train_df, test_size=0.3, random_state=42, stratify=train_df['diagnosis']
)
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['diagnosis']
)

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")
print(f"Test: {len(test_data)} samples")

# Reset index
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

In [ ]:
# Calculate class weights for imbalanced data
class_counts = np.bincount(train_data['diagnosis'].values)
class_weights = len(train_data) / (config.num_classes * class_counts)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f"Class weights: {class_weights}")

## 7. Dataset & Augmentations

In [ ]:
# Data augmentation
def get_train_transforms(image_size):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
        A.GaussNoise(p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(image_size):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

In [ ]:
# Dataset class
class RetinalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = row['diagnosis']
        
        if self.transform:
            image = self.transform(image=image)['image']
        
        return image, label

In [ ]:
# Create datasets and dataloaders
train_dataset = RetinalDataset(train_data, get_train_transforms(config.image_size))
val_dataset = RetinalDataset(val_data, get_val_transforms(config.image_size))
test_dataset = RetinalDataset(test_data, get_val_transforms(config.image_size))

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 8. Model Architecture

In [ ]:
# Model factory
def create_model(model_name, num_classes, pretrained=True, dropout=0.2):
    if 'vit' in model_name:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    elif 'resnet' in model_name:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    elif 'efficientnet' in model_name:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    else:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

# Create model
model = create_model(config.model_name, config.num_classes, config.pretrained, config.dropout)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {config.model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 9. Loss Function (Focal Loss)

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def forward(self, inputs, targets):
        num_classes = inputs.shape[1]
        
        # Label smoothing
        targets_one_hot = F.one_hot(targets, num_classes).float()
        if self.label_smoothing > 0:
            targets_one_hot = targets_one_hot * (1 - self.label_smoothing) + self.label_smoothing / num_classes
        
        # Compute probabilities
        p = F.softmax(inputs, dim=1)
        
        # Focal weight
        focal_weight = (1 - p) ** self.gamma
        
        # Cross entropy
        ce = -targets_one_hot * torch.log(p.clamp(min=1e-8))
        
        # Apply focal weight
        loss = focal_weight * ce
        
        # Apply class weights
        if self.alpha is not None:
            loss = self.alpha.unsqueeze(0) * loss
        
        return loss.sum(dim=1).mean()

# Create loss function
criterion = FocalLoss(gamma=config.focal_gamma, alpha=class_weights, label_smoothing=config.label_smoothing)

## 10. Mixup Augmentation

In [ ]:
def mixup_data(x, y, alpha=0.2):
    """Apply Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute mixup loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 11. Training Setup

In [ ]:
# Optimizer with layer-wise learning rate decay for ViT
def get_optimizer(model, lr, weight_decay):
    # No weight decay for bias and normalization layers
    no_decay = ['bias', 'LayerNorm.weight', 'norm']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': weight_decay,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    return torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)

optimizer = get_optimizer(model, config.learning_rate, config.weight_decay)

# Cosine annealing scheduler with warmup
num_training_steps = len(train_loader) * config.num_epochs
num_warmup_steps = len(train_loader) * config.warmup_epochs

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

# Mixed precision scaler
scaler = GradScaler()

print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

## 12. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, use_mixup=True):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Mixup
        if use_mixup and config.mixup_alpha > 0:
            images, labels_a, labels_b, lam = mixup_data(images, labels, config.mixup_alpha)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            if use_mixup and config.mixup_alpha > 0:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
        
        # Backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        
        if not (use_mixup and config.mixup_alpha > 0):
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    avg_loss = total_loss / len(loader)
    
    if len(all_preds) > 0:
        acc = accuracy_score(all_labels, all_preds)
    else:
        acc = 0.0
    
    return avg_loss, acc

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, labels in tqdm(loader, desc='Evaluating'):
        images, labels = images.to(device), labels.to(device)
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        total_loss += loss.item()
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds),
        'balanced_accuracy': balanced_accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro'),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted'),
        'kappa': cohen_kappa_score(all_labels, all_preds, weights='quadratic'),
    }
    
    return metrics, all_preds, all_labels, all_probs

In [ ]:
# Training
best_val_acc = 0
best_val_kappa = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_kappa': [], 'lr': []}

print(f"\nStarting training for {config.num_epochs} epochs...\n")

for epoch in range(config.num_epochs):
    print(f"\nEpoch {epoch+1}/{config.num_epochs}")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler)
    
    # Validate
    val_metrics, _, _, _ = evaluate(model, val_loader, criterion)
    
    # Log metrics
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_kappa'].append(val_metrics['kappa'])
    history['lr'].append(scheduler.get_last_lr()[0])
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
          f"Balanced Acc: {val_metrics['balanced_accuracy']:.4f} | Kappa: {val_metrics['kappa']:.4f}")
    
    # Save best model
    if val_metrics['kappa'] > best_val_kappa:
        best_val_kappa = val_metrics['kappa']
        best_val_acc = val_metrics['accuracy']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
        }, 'best_model.pth')
        print(f"  Saved best model! (Kappa: {best_val_kappa:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= config.patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete!")
print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Best Val Kappa: {best_val_kappa:.4f}")

## 13. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy & Kappa
axes[1].plot(history['val_acc'], label='Accuracy')
axes[1].plot(history['val_kappa'], label='Kappa')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(history['lr'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 14. Final Evaluation on Test Set

In [ ]:
# Load best model
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_metrics, test_preds, test_labels, test_probs = evaluate(model, test_loader, criterion)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy:          {test_metrics['accuracy']:.4f}")
print(f"Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
print(f"F1 (Macro):        {test_metrics['f1_macro']:.4f}")
print(f"F1 (Weighted):     {test_metrics['f1_weighted']:.4f}")
print(f"Quadratic Kappa:   {test_metrics['kappa']:.4f}")
print("="*50)

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=config.class_names, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=config.class_names,
            yticklabels=config.class_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', xticklabels=config.class_names,
            yticklabels=config.class_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 15. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarize labels
test_labels_bin = label_binarize(test_labels, classes=list(range(config.num_classes)))

# Plot ROC curves
plt.figure(figsize=(10, 8))
colors = plt.cm.Set1(np.linspace(0, 1, config.num_classes))

for i, (class_name, color) in enumerate(zip(config.class_names, colors)):
    fpr, tpr, _ = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{class_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - One vs Rest')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.savefig('roc_curves.png', dpi=150)
plt.show()

## 16. Save Model for Deployment

In [ ]:
# Save final model
torch.save({
    'model_name': config.model_name,
    'num_classes': config.num_classes,
    'model_state_dict': model.state_dict(),
    'class_names': config.class_names,
    'test_metrics': test_metrics,
}, f'{config.model_name}_final.pth')

print(f"Model saved as {config.model_name}_final.pth")

In [ ]:
# Download files
from google.colab import files

files.download('best_model.pth')
files.download(f'{config.model_name}_final.pth')
files.download('training_history.png')
files.download('confusion_matrix.png')
files.download('roc_curves.png')

## 17. Train Other Models (Optional)

To train ResNet-50 or EfficientNet-B4, change `config.model_name` and re-run from Section 8.

In [ ]:
# Uncomment to train different models:

# config.model_name = 'resnet50'
# config.model_name = 'efficientnet_b4'

# Then re-run cells from Section 8 onwards